# Delta Lake com Apache Spark — Mercado de Trabalho em IA

## Cenário

Este notebook analisa o dataset **AI Job Market Insights** (500 registros de vagas em IA) e demonstra as capacidades **ACID** e **Time Travel** do Delta Lake.

## Operações Demonstradas

1. **Carga Inicial** — lê o CSV e cria a tabela
2. **INSERT** — adiciona novos registros
3. **UPDATE** — modifica registros existentes
4. **DELETE** — remove registros
5. **DESCRIBE HISTORY** — inspeciona o histórico de versões
6. **Time Travel** — consulta dados em versões anteriores

## Configuração Inicial

Inicializa a SparkSession com suporte a Delta Lake. Remove artefatos de execuções anteriores para garantir um estado limpo.

In [1]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession

project_root = Path.cwd()
data_path = str(project_root / "data" / "ai_job_market_insights.csv")
warehouse_dir = project_root / "spark-warehouse"

delta_path = warehouse_dir / "ai_jobs_delta"
if delta_path.exists():
    shutil.rmtree(delta_path)

spark = (
    SparkSession.builder
    .appName("Delta-Lake-AI-Jobs")
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(warehouse_dir.resolve()))
    .getOrCreate()
)

spark.sql("DROP TABLE IF EXISTS ai_jobs_delta")
spark

26/04/26 15:55:05 WARN Utils: Your hostname, Xande resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/26 15:55:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/alexandre/Apache-spark/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/alexandre/.ivy2/cache
The jars for the packages stored in: /home/alexandre/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6ef9b888-ac99-411e-bafa-30998160fbd4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 104ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   

## Carga Inicial

Lê o CSV e cria a tabela Delta `ai_jobs_delta` com os 500 registros de vagas em IA.

In [2]:
df = spark.read.option("header", True).option("inferSchema", True).csv(data_path)
df.createOrReplaceTempView("ai_jobs_src")

spark.sql("""
CREATE OR REPLACE TABLE ai_jobs_delta (
    Job_Title             STRING,
    Industry              STRING,
    Company_Size          STRING,
    Location              STRING,
    AI_Adoption_Level     STRING,
    Automation_Risk       STRING,
    Required_Skills       STRING,
    Salary_USD            DOUBLE,
    Remote_Friendly       STRING,
    Job_Growth_Projection STRING
) USING DELTA
""")

spark.sql("INSERT INTO ai_jobs_delta SELECT * FROM ai_jobs_src")

total = spark.sql("SELECT COUNT(*) FROM ai_jobs_delta").collect()[0][0]
print(f"✓ Registros carregados: {total}")
spark.sql("SELECT * FROM ai_jobs_delta LIMIT 3").show(truncate=False)

26/04/26 15:56:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

✓ Registros carregados: 500
+---------------------+-------------+------------+---------+-----------------+---------------+---------------+------------------+---------------+---------------------+
|Job_Title            |Industry     |Company_Size|Location |AI_Adoption_Level|Automation_Risk|Required_Skills|Salary_USD        |Remote_Friendly|Job_Growth_Projection|
+---------------------+-------------+------------+---------+-----------------+---------------+---------------+------------------+---------------+---------------------+
|Cybersecurity Analyst|Entertainment|Small       |Dubai    |Medium           |High           |UX/UI Design   |111392.16524315962|Yes            |Growth               |
|Marketing Specialist |Technology   |Large       |Singapore|Medium           |High           |Marketing      |93792.56246610906 |No             |Decline              |
|AI Researcher        |Technology   |Large       |Singapore|Medium           |High           |UX/UI Design   |107170.26306894995|Yes

## 🔹 ACID: INSERT

Adiciona um novo registro — vaga de Data Engineer em Urussanga, SC (Brasil).

In [3]:
spark.sql("""
INSERT INTO ai_jobs_delta VALUES
  ('Data Engineer', 'Technology', 'Medium', 'Urussanga',
   'High', 'Low', 'Python,Spark,SQL', 95000.00, 'Yes', 'Growth')
""")

print("✓ INSERT executado. Nova vaga em Urussanga:")
spark.sql("SELECT * FROM ai_jobs_delta WHERE Location = 'Urussanga'").show(truncate=False)

✓ INSERT executado. Nova vaga em Urussanga:
+-------------+----------+------------+---------+-----------------+---------------+----------------+----------+---------------+---------------------+
|Job_Title    |Industry  |Company_Size|Location |AI_Adoption_Level|Automation_Risk|Required_Skills |Salary_USD|Remote_Friendly|Job_Growth_Projection|
+-------------+----------+------------+---------+-----------------+---------------+----------------+----------+---------------+---------------------+
|Data Engineer|Technology|Medium      |Urussanga|High             |Low            |Python,Spark,SQL|95000.0   |Yes            |Growth               |
+-------------+----------+------------+---------+-----------------+---------------+----------------+----------+---------------+---------------------+



## 🔹 ACID: UPDATE

Reajusta o salário em +10% para todas as vagas com alto nível de adoção de IA E baixo risco de automação.

In [4]:
spark.sql("""
UPDATE ai_jobs_delta
SET Salary_USD = ROUND(Salary_USD * 1.10, 2)
WHERE AI_Adoption_Level = 'High' AND Automation_Risk = 'Low'
""")

print("✓ UPDATE executado. Amostra de vagas reajustadas:")
spark.sql("""
SELECT Job_Title, Location, AI_Adoption_Level, Automation_Risk, Salary_USD
FROM ai_jobs_delta
WHERE AI_Adoption_Level = 'High' AND Automation_Risk = 'Low'
LIMIT 4
""").show(truncate=False)

✓ UPDATE executado. Amostra de vagas reajustadas:
+-------------+---------+-----------------+---------------+----------+
|Job_Title    |Location |AI_Adoption_Level|Automation_Risk|Salary_USD|
+-------------+---------+-----------------+---------------+----------+
|AI Researcher|London   |High             |Low            |82517.45  |
|Sales Manager|Singapore|High             |Low            |106518.04 |
|Sales Manager|Dubai    |High             |Low            |91079.29  |
|HR Manager   |Tokyo    |High             |Low            |80655.63  |
+-------------+---------+-----------------+---------------+----------+



## 🔹 ACID: DELETE

Remove todas as vagas com perspectiva de declínio (`Job_Growth_Projection = 'Decline'`).

In [5]:
antes = spark.sql("SELECT COUNT(*) FROM ai_jobs_delta").collect()[0][0]
spark.sql("DELETE FROM ai_jobs_delta WHERE Job_Growth_Projection = 'Decline'")
depois = spark.sql("SELECT COUNT(*) FROM ai_jobs_delta").collect()[0][0]

removidos = antes - depois
print(f"✓ DELETE executado.")
print(f"  Antes: {antes} | Depois: {depois} | Removidos: {removidos}")

✓ DELETE executado.
  Antes: 501 | Depois: 332 | Removidos: 169


## Histórico de Versões

Exibe o log completo de todas as operações realizadas na tabela. Cada versão corresponde a uma operação (CREATE, INSERT, UPDATE, DELETE).

In [6]:
spark.sql("DESCRIBE HISTORY ai_jobs_delta").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+-----------------------+----------------------------------------------------------------------------------------------+
|version|timestamp              |operation              |operationParameters                                                                           |
+-------+-----------------------+-----------------------+----------------------------------------------------------------------------------------------+
|4      |2026-04-26 15:57:07.597|DELETE                 |{predicate -> ["(Job_Growth_Projection#3594 = Decline)"]}                                     |
|3      |2026-04-26 15:57:02.3  |UPDATE                 |{predicate -> ["((AI_Adoption_Level#2076 = High) AND (Automation_Risk#2077 = Low))"]}         |
|2      |2026-04-26 15:56:54.566|WRITE                  |{mode -> Append, partitionBy -> []}                                                           |
|1      |2026-04-26 15:56:47.615|WRITE                  |{mode -> Append, partitio

## Time Travel

Volta no tempo e consulta a tabela na **Versão 0** — o estado original, imediatamente após a carga inicial (antes de INSERT, UPDATE e DELETE).

In [7]:
total_v0 = spark.sql("SELECT COUNT(*) FROM ai_jobs_delta VERSION AS OF 0").collect()[0][0]
print(f"✓ Time Travel — Versão 0")
print(f"  Total de registros: {total_v0}")
print("\n  Primeiros 3 registros do estado original:")
spark.sql("SELECT * FROM ai_jobs_delta VERSION AS OF 0 LIMIT 3").show(truncate=False)

✓ Time Travel — Versão 0
  Total de registros: 0

  Primeiros 3 registros do estado original:
+---------+--------+------------+--------+-----------------+---------------+---------------+----------+---------------+---------------------+
|Job_Title|Industry|Company_Size|Location|AI_Adoption_Level|Automation_Risk|Required_Skills|Salary_USD|Remote_Friendly|Job_Growth_Projection|
+---------+--------+------------+--------+-----------------+---------------+---------------+----------+---------------+---------------------+
+---------+--------+------------+--------+-----------------+---------------+---------------+----------+---------------+---------------------+



In [8]:
spark.stop()
print("✓ SparkSession encerrada.")

✓ SparkSession encerrada.
